# WWF ITR Tool — Standalone portfolio coverage

Calculate the investment-weighted share of your portfolio with **SBTi-validated near-term targets**, using pandas and the SBTi Companies Taking Action (CTA) file. **The `wwf-itr` package is not required.**

This notebook needs only a portfolio file and CTA data. It does **not** calculate temperature scores or require company emissions, financial fundamentals, or target data. It uses WATS weighting only.

**Coverage is based on the status in the CTA file used for this run. It does not reconstruct historical SBTi status.** A historical portfolio date alone cannot establish which targets were validated at that date.

Run the cells in order. Start with the sample portfolio, then replace it with your own file. For confidential portfolio data, run Jupyter locally rather than uploading it to Google Colab.

See the [shared documentation](https://wwf-sweden.github.io/ITR-tool/index.html), [data requirements](https://wwf-sweden.github.io/ITR-tool/DataRequirements.html), and [CDP-WWF methodology](https://wwfint.awsassets.panda.org/downloads/cdp-wwf-temperature-scoring-methodology---september-2024.pdf).

**Sample data:** this notebook uses `example_portfolio_360.csv` and, where needed, `data_provider_example_360.xlsx` from `examples/data`. The 360 companies, financial figures and targets are fictional; public LEI/ISIN identifiers are reused for illustrative CTA matching. Run all cells again after switching datasets. Download links target `main` and become available when these files are published there; when testing a published review branch, replace `main` in the download URL with that branch.


## 1. Install and import

Use Python 3.10 or later. Only pandas, requests and Excel-reading libraries are needed; no temperature-scoring package is installed. Run this installation cell in the notebook's kernel; restart the kernel if an already imported package is upgraded.

In [ ]:
%pip install "pandas>=2.2,<3" "requests>=2.32,<3" "openpyxl>=3.1,<4" "xlrd>=2,<3"

In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
from io import BytesIO
from math import isfinite
from pathlib import Path
import urllib.request

import pandas as pd
import requests
from IPython.display import display

print(f"pandas version: {pd.__version__}")
print(f"requests version: {requests.__version__}")

## 2. Choose your portfolio

The sample comes from WWF-Sweden/ITR-tool and is for demonstration only. Existing files are not overwritten.

For your own data, set `portfolio_path` to a CSV or Excel file. In Colab, upload a non-sensitive file using the Files panel. CSV files should use comma separators; change `csv_encoding` if necessary. Excel files are read from their first sheet.

Required columns:

| Column | Meaning |
| --- | --- |
| `company_name` | Company name for display |
| `company_id` | Unique company identifier; use one row per company in this simplified notebook |
| `investment_value` | Non-negative holding value; use the same currency for all rows |
| `company_lei` and/or `company_isin` | Identifier used to match SBTi status |

One identifier column is sufficient. Individual rows may have neither identifier, but those holdings remain in the denominator and do not count as validated. Missing matches do not prove that a company has no validated targets.

The package uses LEI when available (more than three characters); otherwise it uses ISIN. It does not retry ISIN after an unmatched LEI. Check identifiers carefully.

In [ ]:
portfolio_path = Path("data/example_portfolio_360.csv")
csv_encoding = "utf-8-sig"

# Download only the default sample; custom paths must already exist.
if portfolio_path == Path("data/example_portfolio_360.csv") and not portfolio_path.exists():
    portfolio_path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/WWF-Sweden/ITR-tool/main/examples/data/example_portfolio_360.csv",
        portfolio_path,
    )

if portfolio_path.suffix.lower() == ".csv":
    portfolio = pd.read_csv(
        portfolio_path, encoding=csv_encoding,
        dtype={"company_id": "string", "company_isin": "string", "company_lei": "string"},
    )
elif portfolio_path.suffix.lower() in {".xlsx", ".xls"}:
    portfolio = pd.read_excel(
        portfolio_path,
        dtype={"company_id": "string", "company_isin": "string", "company_lei": "string"},
    )
else:
    raise ValueError("Choose a .csv, .xlsx or .xls portfolio file.")

In [ ]:
required_columns = {"company_name", "company_id", "investment_value"}
missing = sorted(required_columns - set(portfolio.columns))
if missing:
    raise ValueError(f"Missing portfolio columns: {', '.join(missing)}")
if portfolio.empty:
    raise ValueError("The portfolio must contain at least one company.")
if not {"company_isin", "company_lei"}.intersection(portfolio.columns):
    raise ValueError("Include company_isin and/or company_lei for SBTi matching.")

portfolio = portfolio.copy()
for column in ("company_name", "company_id", "company_isin", "company_lei"):
    if column not in portfolio.columns:
        portfolio[column] = None
    portfolio[column] = portfolio[column].astype("string").str.strip().replace("", pd.NA)

if portfolio[["company_name", "company_id"]].isna().any().any():
    raise ValueError("Company names and company IDs must not be empty.")
if portfolio["company_id"].duplicated().any():
    raise ValueError("Use one row per company_id; combine holdings in the same company first.")

portfolio["investment_value"] = pd.to_numeric(portfolio["investment_value"], errors="raise")
values = portfolio["investment_value"]
total_investment = float(values.sum())
if (
    not values.map(isfinite).all()
    or (values < 0).any()
    or not isfinite(total_investment)
    or total_investment <= 0
):
    raise ValueError("Investment values must be finite, non-negative, and sum to more than zero.")

# Always resolve status from this run's CTA file, ignoring previously saved status columns.
portfolio = portfolio[
    ["company_name", "company_id", "company_isin", "company_lei", "investment_value"]
].copy()
portfolio[["company_isin", "company_lei"]] = portfolio[
    ["company_isin", "company_lei"]
].astype(object).where(portfolio[["company_isin", "company_lei"]].notna(), None)

has_lei = portfolio["company_lei"].fillna("").str.len().gt(3)
has_isin = portfolio["company_isin"].fillna("").str.len().gt(0)
print(f"Loaded {len(portfolio)} companies from {portfolio_path}")
print(f"Holdings without usable LEI or ISIN: {(~(has_lei | has_isin)).sum()}")
display(portfolio.head())

## 3. Choose and load CTA data

Leave `cta_file_path = None` to download the current CTA file directly from SBTi. This requires internet access and downloads a new copy each time you run the cell. There is no automatic cache or bundled fallback: a failed download raises an error rather than silently using old data.

For offline or reproducible analysis, set `cta_file_path` to a saved CTA Excel file. The current format requires `company_name`, `isin`, `lei`, and `near_term_status`. This notebook does not read the old `Action` / `Target` / `Date Published` format.

After validation, downloaded data is saved to `data/cta_download.xlsx`, replacing the previous download there. Keep a separately named copy with your analysis when you need to retain a snapshot. The source and SHA-256 fingerprint are printed to identify the exact input file; a file modification time is not a historical SBTi status date.

In [ ]:
cta_file_path = None  # Example: "data/my-cta-snapshot.xlsx"
cta_url = "https://sb-ti.files.svdcdn.com/staging/files/companies-excel.xlsx"

if cta_file_path is None:
    response = requests.get(
        cta_url,
        headers={"x-request": "download", "User-Agent": "WWF-ITR-portfolio-coverage-notebook"},
        timeout=30,
    )
    response.raise_for_status()
    cta_bytes = response.content
    cta_source = cta_url
    print(f"Downloaded at: {datetime.now(timezone.utc).isoformat()}")
else:
    cta_path = Path(cta_file_path)
    cta_bytes = cta_path.read_bytes()
    cta_source = str(cta_path.resolve())

# Keep identifiers as strings so leading zeros are preserved.
cta = pd.read_excel(BytesIO(cta_bytes), dtype={"isin": "string", "lei": "string"})
print(f"CTA source: {cta_source}")
print(f"CTA SHA-256: {sha256(cta_bytes).hexdigest()}")

## 4. Validate CTA status labels

`Validated Targets` counts as validated; the previous label `Targets set` is also accepted. `Commitment to Set Targets`, `Committed`, `Commitment Removed` and empty statuses do not count. Matching ignores capitalization and surrounding whitespace in status labels. Unknown nonempty statuses or missing required columns raise an error so a changed file format cannot silently become zero coverage.

Only validated companies are used for identifier matching. The notebook follows the current tool's LEI-first rule: when a portfolio row has a usable LEI, it does not retry ISIN after an unmatched LEI.

In [ ]:
required_cta_columns = {"company_name", "isin", "lei", "near_term_status"}
missing = sorted(required_cta_columns - set(cta.columns))
if missing:
    raise ValueError(f"CTA file is missing required columns: {', '.join(missing)}")
if cta.empty:
    raise ValueError("The CTA file contains no company rows.")

status = cta["near_term_status"].astype("string").str.strip().str.casefold().fillna("")
status_is_validated = {
    "validated targets": True,
    "targets set": True,
    "commitment to set targets": False,
    "committed": False,
    "commitment removed": False,
    "": False,
}
unknown = ~status.isin(status_is_validated)
if unknown.any():
    labels = sorted(cta.loc[unknown, "near_term_status"].astype(str).unique())
    raise ValueError(f"Unknown CTA near-term status values: {', '.join(labels)}")

cta_validated = cta.loc[status.map(status_is_validated)].copy()
for column in ("isin", "lei"):
    cta_validated[column] = cta_validated[column].astype("string").str.strip().replace("", pd.NA)

if cta_file_path is None:
    snapshot_path = Path("data/cta_download.xlsx")
    snapshot_path.parent.mkdir(parents=True, exist_ok=True)
    snapshot_path.write_bytes(cta_bytes)
    print(f"Saved validated CTA download to {snapshot_path.resolve()}")

print(f"CTA rows: {len(cta)}; rows with validated near-term targets: {len(cta_validated)}")

## 5. Calculate investment-weighted coverage

The result is a percentage from **0 to 100**, calculated directly as:

`100 × investment value with validated targets / total portfolio investment value`

Holdings without identifiers or a match remain in the denominator and contribute zero to the numerator. An unmatched holding is not proof that the company has no validated targets. Check identifier quality and the CTA snapshot before interpreting the result.

In [ ]:
coverage_data = portfolio.copy()
validated_leis = set(cta_validated["lei"].dropna())
validated_isins = set(cta_validated["isin"].dropna())

lei = coverage_data["company_lei"].astype("string")
isin = coverage_data["company_isin"].astype("string")
use_lei = lei.notna() & ~lei.str.casefold().isin(["none", "nan"]) & lei.str.len().gt(3).fillna(False)
use_isin = ~use_lei & isin.notna() & ~isin.str.casefold().isin(["none", "nan"])

coverage_data["sbti_validated"] = (
    (use_lei & lei.isin(validated_leis)) | (use_isin & isin.isin(validated_isins))
)
coverage = 100 * (
    coverage_data.loc[coverage_data["sbti_validated"], "investment_value"].sum()
    / coverage_data["investment_value"].sum()
)
print(f"Portfolio coverage (WATS): {coverage:.2f}%")

## 6. Inspect and export

`sbti_validated` shows whether a holding matched a validated company in the selected CTA data. `coverage_contribution_pp` is the holding's contribution in **percentage points**; these contributions sum to the displayed portfolio coverage.

Keep the portfolio and CTA snapshot, the notebook, and the printed library versions with your analysis if you need to reproduce it later. The CSV contains portfolio information: store it appropriately.


In [ ]:
coverage_data["portfolio_weight_pct"] = (
    100 * coverage_data["investment_value"] / coverage_data["investment_value"].sum()
)
coverage_data["coverage_contribution_pp"] = (
    coverage_data["portfolio_weight_pct"] * coverage_data["sbti_validated"].astype(int)
)
display(coverage_data)

output_path = Path("data/portfolio_coverage_results.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
coverage_data.to_csv(output_path, index=False)
print(f"Saved portfolio coverage details to {output_path.resolve()}")